# Step 0 — NFP reconstruction for all 4 datasets

Single-GPU notebook version of `step0.py`. Reads parameters from `config_step0.conf`.

For each dataset: reads scan data, runs BH iterative reconstruction (RecNFP), saves probe/projection/position results to a combined HDF5.

In [ ]:
import json
import os
import numpy as np
import cupy as cp
import h5py
from types import SimpleNamespace
from mpi4py import MPI
from holotomocupy.rec_nfp_mpi import RecNFP
from holotomocupy.config import parse_args_step0
from holotomocupy.utils import *

In [ ]:
config_file = 'config_step0.conf'
args = parse_args_step0(config_file)
logger.setLevel(args.log_level)

comm = MPI.COMM_SELF  # single-GPU notebook

print(f'dataset_ids = {args.dataset_ids}')
print(f'n={args.n}  niter={args.niter}  nchunk={args.nchunk}  vis_step={args.vis_step}')
print(f'h5_out={args.h5_out}')

## Geometry helpers

In [ ]:
def _read_h5_field(h5path, suffix):
    result = {}
    def _visit(name, obj):
        if not result and isinstance(obj, h5py.Dataset) and name.endswith(suffix):
            result['val'] = obj[()]
    with h5py.File(h5path, 'r') as f:
        f.visititems(_visit)
    if not result:
        raise KeyError(f'{suffix!r} not found in {h5path}')
    return result['val']

def read_energy(h5path):
    return float(_read_h5_field(h5path, 'TOMO/energy'))

def read_sx0(h5path):
    return float(_read_h5_field(h5path, 'TOMO/sx0')) * 1e-3

def read_sx(h5path):
    names  = _read_h5_field(h5path, 'sample/positioners/name').decode().split()
    values = _read_h5_field(h5path, 'sample/positioners/value').decode().split()
    return float(values[names.index('sx')]) * 1e-3

def read_detector_pixelsize(h5path):
    par = json.loads(_read_h5_field(h5path, 'TOMO/FTOMO_PAR').decode())
    return float(par['image_pixel_size']) * 1e-6

def read_focustodetectordistance(h5path):
    return float(_read_h5_field(h5path, 'PTYCHO/focusToDetectorDistance')) * 1e-3

## Reconstruct each dataset

In [ ]:

_ref_id = str(args.dataset_ids[0])

prb_amps    = []
prb_phases  = []
proj_deltas = []
proj_betas  = []
pos_errs    = []

for dataset_id in args.dataset_ids:
    _sid      = str(dataset_id)
    scan_file = args.scan_file.replace(f'_{_ref_id}_', f'_{_sid}_')
    meta_file = args.meta_file.replace(f'_{_ref_id}_', f'_{_sid}_')

    print(f'=== Dataset {dataset_id} ===')

    # --- Geometry ---
    energy                  = read_energy(meta_file)
    z1_ref                  = read_sx0(meta_file)
    z1                      = read_sx(meta_file) - z1_ref
    detector_pixelsize      = read_detector_pixelsize(meta_file)
    focustodetectordistance = read_focustodetectordistance(meta_file)

    magnification = focustodetectordistance / z1
    voxelsize     = detector_pixelsize / magnification
    print(f'energy={energy} keV  z1={z1*1e3:.4f} mm  mag={magnification:.2f}  voxel={voxelsize*1e9:.2f} nm')

    # --- Positions and data dimensions ---
    with h5py.File(scan_file, 'r') as f:
        dset    = f['entry_0000/ESRF-ID16A/PCIe/data']
        ntheta, ny, nx = dset.shape
        spy_str = f['entry_0000/ESRF-ID16A/PCIe/header/spy'][()]
        spz_str = f['entry_0000/ESRF-ID16A/PCIe/header/spz'][()]

    n   = args.n
    sty = (ny - n) // 2
    stx = (nx - n) // 2

    spy = np.array(spy_str.decode().split(), dtype='float32') * 1e-6
    spz = np.array(spz_str.decode().split(), dtype='float32') * 1e-6
    pos = np.stack([-spz / voxelsize, spy / voxelsize], axis=-1).astype('float32')
    print(f'pos (pix): y in [{pos[:,0].min():.2f}, {pos[:,0].max():.2f}]  x in [{pos[:,1].min():.2f}, {pos[:,1].max():.2f}]')

    pos_range = int(np.ceil(np.abs(pos).max())) + 8
    nobj      = int(np.ceil((n + 2 * pos_range) / 32)) * 32
    print(f'n={n}  nobj={nobj}  pos_range=\xb1{pos_range} pix')

    # --- Init RecNFP ---
    rec_args = SimpleNamespace(
        energy                  = energy,
        detector_pixelsize      = detector_pixelsize,
        focustodetectordistance = focustodetectordistance,
        z1                      = z1,
        ntheta                  = ntheta,
        nz                      = n,
        n                       = n,
        nzobj                   = nobj,
        nobj                    = nobj,
        obj_dtype               = 'complex64',
        rho                     = args.rho,
        niter                   = args.niter,
        nchunk                  = args.nchunk,
        vis_step                = args.vis_step,
        err_step                = args.err_step,
        start_iter              = 0,
        comm                    = comm,
    )

    cl = RecNFP(rec_args)

    # --- Load data ---
    with h5py.File(scan_file, 'r') as f:
        raw_slice = f['entry_0000/ESRF-ID16A/PCIe/data'][cl.st_theta:cl.end_theta, sty:sty+n, stx:stx+n].astype('float32')
    global_mean = comm.allreduce(raw_slice.sum(), op=MPI.SUM) / (ntheta * n * n)
    cl.data[:] = np.sqrt(np.abs(raw_slice / (global_mean + 1e-5)))

    cl.vars['proj'][:] = 0
    cl.vars['prb'][:]  = 1
    cl.vars['pos'][:]  = cp.array(pos[cl.st_theta:cl.end_theta])

    # --- Reconstruct ---
    cl.BH()

    # --- Collect results ---
    pos_final = cl.vars['pos'].get()
    pos_init  = cl.pos_init.get()
    pos_err   = pos_final - pos_init
    print(f'pos err  y: max={np.abs(pos_err[:,0]).max():.4f}  x: max={np.abs(pos_err[:,1]).max():.4f}')

    prb_np  = cl.vars['prb'].get()
    proj_np = cl.vars['proj'].get()
    prb_amps.append(np.abs(prb_np))
    prb_phases.append(np.angle(prb_np))
    proj_deltas.append(proj_np.real)
    proj_betas.append(proj_np.imag)
    pos_errs.append(pos_err)

    del cl
    cp.get_default_memory_pool().free_all_blocks()

print('All datasets done.')

## Save combined HDF5

In [ ]:
os.makedirs(os.path.dirname(os.path.abspath(args.h5_out)), exist_ok=True)
with h5py.File(args.h5_out, 'w') as f:
    f.create_dataset('prb_amp',    data=np.stack(prb_amps))
    f.create_dataset('prb_phase',  data=np.stack(prb_phases))
    f.create_dataset('proj_delta', data=np.stack(proj_deltas))
    f.create_dataset('proj_beta',  data=np.stack(proj_betas))
    f.create_dataset('pos_err',    data=np.concatenate(pos_errs, axis=0))
print(f'Saved to {args.h5_out}')

## Visualise results

In [ ]:
import matplotlib.pyplot as plt

show = True
if show:
    nd  = len(args.dataset_ids)
    fig, axes = plt.subplots(3, nd, figsize=(4*nd, 10))
    if nd == 1:
        axes = axes[:, None]
    for i, did in enumerate(args.dataset_ids):
        amp = prb_amps[i]
        ph  = prb_phases[i]
        err = pos_errs[i]
        axes[0, i].imshow(amp, cmap='gray')
        axes[0, i].set_title(f'prb amp  ds{did}')
        axes[0, i].axis('off')
        axes[1, i].imshow(ph, cmap='RdBu_r')
        axes[1, i].set_title(f'prb phase  ds{did}')
        axes[1, i].axis('off')
        axes[2, i].plot(err[:, 0], label='y')
        axes[2, i].plot(err[:, 1], label='x')
        axes[2, i].set_title(f'pos err (pix)  ds{did}')
        axes[2, i].legend()
    plt.tight_layout()
    plt.show()